# Maize (Corn) Grain Yield Prediction Dataset: Exploratory Data Analysis & Quality Audit (2002–2025)

**Principal Investigator / Research Role:** Senior Agricultural Data Scientist & Machine Learning Researcher  
**Target Variable:** USDA NASS County Corn Grain Yield (`corn_yield_bu_acre`)  
**Geographic Coverage:** 9 US Corn Belt States (IA, IL, IN, MN, MO, NE, OH, SD, WI) — 814 Counties  
**Temporal Domain:** 2002–2025 Labeled Yield Observations; 2026 Unlabeled Prospective Forecasting Set  
**Primary Data Sources:**
- **Yield & Phenology:** USDA National Agricultural Statistics Service (NASS) QuickStats
- **Climate & Gridded Weather:** NOAA NCEI Climate Division / County Grids & Oregon State PRISM Climate Group
- **Soil Profiles:** ISRIC SoilGrids 2.0 (Clay, Sand, Silt, SOC, pH, Bulk Density across 0–30 cm)
- **Crop Management:** USDA Risk Management Agency (RMA) Summary of Business & NASS Chemical Use Surveys


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure visualization standards
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 150

print("Libraries imported successfully.")


## 1. Load Integrated Research Datasets
We load the three integrated dataset products generated by the data engineering pipeline:
1. `County_Year_Master_Dataset.csv`: Full 20,350 row matrix with all domain variables and diagnostics.
2. `County_Year_Feature_Table.csv`: 16,390 row ML-ready supervised dataset (2002–2025) with authentic yield labels and zero post-harvest leakage.
3. `Future_2026_Feature_Table.csv`: 814 row prospective unlabeled dataset for 2026 harvest prediction.


In [ ]:
DATA_DIR = r"E:\ml do not open\Maize_Yield_Research_2002_2026_Integrated_Data"

master_df = pd.read_csv(os.path.join(DATA_DIR, "County_Year_Master_Dataset.csv"), low_memory=False)
feat_df = pd.read_csv(os.path.join(DATA_DIR, "County_Year_Feature_Table.csv"), low_memory=False)
future_df = pd.read_csv(os.path.join(DATA_DIR, "Future_2026_Feature_Table.csv"), low_memory=False)

print(f"Master Dataset Shape:   {master_df.shape} (2002-2026, 814 counties x 25 years)")
print(f"Feature Table Shape:    {feat_df.shape} (2002-2025 labeled ML training records)")
print(f"Future 2026 Set Shape:  {future_df.shape} (814 prospective counties, strictly unlabeled)")


## 2. Target Variable Inspection: Corn Grain Yield
We evaluate the statistical distribution of county-level corn grain yield (`corn_yield_bu_acre`).


In [ ]:
yield_s = feat_df['corn_yield_bu_acre']
print("=== CORN GRAIN YIELD (BU/ACRE) SUMMARY ===")
print(f"Observation Count: {len(yield_s):,}")
print(f"Mean Yield:        {yield_s.mean():.2f} bu/acre")
print(f"Std Deviation:     {yield_s.std():.2f} bu/acre")
print(f"Median:            {yield_s.median():.2f} bu/acre")
print(f"IQR (Q25 - Q75):   {yield_s.quantile(0.25):.2f} - {yield_s.quantile(0.75):.2f} bu/acre")
print(f"Min / Max:         {yield_s.min():.2f} / {yield_s.max():.2f} bu/acre")
print(f"Skewness:          {yield_s.skew():.3f} (Slight left skew due to drought years)")
print(f"Kurtosis:          {yield_s.kurtosis():.3f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
sns.histplot(yield_s, kde=True, ax=ax1, color='#1f78b4', bins=40, edgecolor='white')
ax1.axvline(yield_s.mean(), color='#e31a1c', linestyle='--', label=f"Mean: {yield_s.mean():.1f}")
ax1.axvline(yield_s.median(), color='#33a02c', linestyle=':', label=f"Median: {yield_s.median():.1f}")
ax1.set_title("Corn Grain Yield Distribution (2002–2025)")
ax1.set_xlabel("Yield (bu/acre)")
ax1.legend()

# Boxplot by State
state_order = feat_df.groupby('state')['corn_yield_bu_acre'].median().sort_values(ascending=False).index
sns.boxplot(data=feat_df, x='state', y='corn_yield_bu_acre', order=state_order, ax=ax2, hue='state', palette='Blues_r', legend=False)
ax2.set_title("Yield Distribution by State")
ax2.set_xlabel("State")
ax2.set_ylabel("Yield (bu/acre)")

plt.tight_layout()
plt.show()


## 3. Annual Yield Progression & Major Climate Shocks (2002–2025)
This section assesses long-term genetic/management yield growth (+1.95 bu/acre/year) alongside acute climate shock events, notably the historic 2012 US Midwest drought.


In [ ]:
annual_yield = feat_df.groupby('year')['corn_yield_bu_acre'].agg(['mean', 'std', 'count']).reset_index()

plt.figure(figsize=(11, 5))
sns.lineplot(data=feat_df, x='year', y='corn_yield_bu_acre', color='#2b5c8f', linewidth=2.5, errorbar=('ci', 95))
plt.scatter(annual_yield['year'], annual_yield['mean'], color='#2b5c8f', s=45, zorder=5)

# Fit trend line
z = np.polyfit(annual_yield['year'], annual_yield['mean'], 1)
p = np.poly1d(z)
plt.plot(annual_yield['year'], p(annual_yield['year']), 'r--', alpha=0.8, label=f"Tech Trend (+{z[0]:.2f} bu/ac/yr)")

# Annotate 2012 drought
d12_val = annual_yield[annual_yield['year'] == 2012]['mean'].values[0]
plt.annotate(f"2012 Drought Shock\nMean: {d12_val:.1f} bu/ac", xy=(2012, d12_val), xytext=(2007, 110),
             arrowprops=dict(facecolor='red', shrink=0.08, width=1.5, headwidth=7),
             fontweight='bold', color='red')

plt.title("Corn Belt Historical Annual Mean Yield Trend & Climate Shocks (2002–2025)")
plt.xlabel("Year")
plt.ylabel("Corn Grain Yield (bu/acre)")
plt.xticks(range(2002, 2026, 2))
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


## 4. Climate Predictors: Rainfall and Reproductive Heat Stress
We analyze the sensitivity of corn yield to cumulative growing season rainfall (May–August) and maximum temperatures during the critical silking/tasseling reproductive phase (July).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Precipitation vs Yield
sns.regplot(data=feat_df, x='noaa_prcp_may_aug_in', y='corn_yield_bu_acre', ax=ax1,
            scatter_kws={'alpha': 0.1, 'color': '#2ca25f', 's': 15}, line_kws={'color': '#006d2c'})
r_p, _ = stats.pearsonr(feat_df['noaa_prcp_may_aug_in'].dropna(), feat_df.loc[feat_df['noaa_prcp_may_aug_in'].notna(), 'corn_yield_bu_acre'])
ax1.set_title(f"Growing Season Rainfall vs Yield (r = {r_p:.2f})")
ax1.set_xlabel("NOAA May–Aug Precipitation (Inches)")
ax1.set_ylabel("Corn Grain Yield (bu/acre)")

# July Heat Stress vs Yield
sns.regplot(data=feat_df, x='noaa_tmax_jul_f', y='corn_yield_bu_acre', ax=ax2,
            scatter_kws={'alpha': 0.1, 'color': '#e34a33', 's': 15}, line_kws={'color': '#b30000'})
r_h, _ = stats.pearsonr(feat_df['noaa_tmax_jul_f'].dropna(), feat_df.loc[feat_df['noaa_tmax_jul_f'].notna(), 'corn_yield_bu_acre'])
ax2.set_title(f"July Maximum Temp vs Yield (r = {r_h:.2f})")
ax2.set_xlabel("NOAA July Mean Max Temperature (°F)")
ax2.set_ylabel("Corn Grain Yield (bu/acre)")

plt.tight_layout()
plt.show()


## 5. Soil Profile Dynamics (ISRIC SoilGrids 2.0)
Soil properties provide static baseline productive capacity. We evaluate the relationship between Soil Organic Carbon (SOC) and historical mean county yield.


In [ ]:
county_summary = feat_df.groupby('county_fips').agg(
    county_mean_yield=('corn_yield_bu_acre', 'mean'),
    soil_soc=('soil_soc_mean_0_30cm', 'first'),
    soil_clay=('soil_clay_mean_0_30cm', 'first'),
    soil_ph=('soil_ph_mean_0_30cm', 'first'),
    state=('state', 'first')
).reset_index()

plt.figure(figsize=(10, 5))
sns.regplot(data=county_summary, x='soil_soc', y='county_mean_yield',
            scatter_kws={'alpha': 0.5, 'color': '#8c510a', 's': 30}, line_kws={'color': '#543005'})
r_soc, _ = stats.pearsonr(county_summary['soil_soc'].dropna(), county_summary.loc[county_summary['soil_soc'].notna(), 'county_mean_yield'])
plt.title(f"County Baseline Soil Organic Carbon (0–30cm) vs 24-Year Mean Yield (r = {r_soc:.2f})")
plt.xlabel("Soil Organic Carbon (dg/kg)")
plt.ylabel("Historical Mean Yield (bu/acre)")
plt.tight_layout()
plt.show()


## 6. Correlation Hierarchy & Feature Collinearity
Evaluation of top linear predictors of corn yield across agricultural condition, climate, soil, and management.


In [ ]:
num_cols = feat_df.select_dtypes(include=[np.number]).columns
corr_yield = feat_df[num_cols].corr()['corn_yield_bu_acre'].drop(['corn_yield_bu_acre', 'cv_percent'], errors='ignore')
corr_top = pd.DataFrame({'feature': corr_yield.index, 'r': corr_yield.values, 'abs_r': corr_yield.abs().values}).sort_values('abs_r', ascending=False).head(15)

plt.figure(figsize=(10, 6))
colors = ['#e41a1c' if x < 0 else '#377eb8' for x in corr_top['r']]
sns.barplot(data=corr_top, y='feature', x='r', hue='feature', palette=colors, legend=False)
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.title("Top 15 Predictor Correlations with Corn Grain Yield")
plt.xlabel("Pearson Correlation (r)")
plt.tight_layout()
plt.show()


## 7. Machine Learning Split Strategy (No Data Leakage)
To prevent temporal data leakage and respect autocorrelated weather patterns, we recommend temporal blocking:
- **Train Set (2002–2019):** 18 calendar years
- **Validation Set (2020–2022):** 3 calendar years (Hyperparameter tuning)
- **Test Set (2023–2025):** 3 calendar years (Held-out benchmark evaluation)
- **Prospective Test (2026):** Unlabeled future prediction period


In [ ]:
train = feat_df[feat_df['year'] <= 2019]
val   = feat_df[(feat_df['year'] >= 2020) & (feat_df['year'] <= 2022)]
test  = feat_df[feat_df['year'] >= 2023]

print(f"Training Set (2002-2019):    {len(train):,} records ({len(train)/len(feat_df)*100:.1f}%)")
print(f"Validation Set (2020-2022):  {len(val):,} records ({len(val)/len(feat_df)*100:.1f}%)")
print(f"Test Set (2023-2025):        {len(test):,} records ({len(test)/len(feat_df)*100:.1f}%)")
print(f"Future Set (2026):           {len(future_df):,} prospective county records")

